# Python / NN / PyTorch 基础实战工作坊

> 目标：把“会看懂”变成“会手写、会调参、会定位问题”。

这个 Notebook 覆盖三层能力：

1. **Python 工程基础**：函数式、装饰器、生成器、数据清洗实践。
2. **神经网络核心机制**：前向/反向传播、梯度检查、优化过程可视化。
3. **PyTorch 训练闭环**：Dataset/DataLoader、模型搭建、训练/评估、常见训练技巧。

建议使用方式：

- 每个代码单元先自己尝试，再运行参考答案。
- 重点关注每段代码后面的“为什么”。
- 最后完成文末练习题，把基础真正吃透。

In [ ]:
import math
import random
from dataclasses import dataclass
from typing import List, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)
print("Torch version:", torch.__version__)

## Part A. Python 基础（面向机器学习工程）

这一部分不是语法大全，而是挑出训练/部署最常见的能力：

- 函数式编程：`map/filter/reduce`、lambda、可组合函数。
- 装饰器：无侵入加日志、计时、重试。
- 生成器：流式读取大文件，控制内存占用。
- 数据处理：用 Pandas 构建训练样本。

In [ ]:
from functools import reduce

nums = [1, 2, 3, 4, 5, 6]

# map: 一对一映射
squares = list(map(lambda x: x * x, nums))

# filter: 条件过滤
evens = list(filter(lambda x: x % 2 == 0, nums))

# reduce: 聚合
sum_all = reduce(lambda a, b: a + b, nums)
product_all = reduce(lambda a, b: a * b, nums)

print("nums      =", nums)
print("squares   =", squares)
print("evens     =", evens)
print("sum       =", sum_all)
print("product   =", product_all)

# 组合式处理：先过滤，再映射，再聚合
pipeline_result = reduce(
    lambda a, b: a + b,
    map(lambda x: x * 10, filter(lambda x: x % 2 == 1, nums)),
)
print("pipeline_result =", pipeline_result)

In [ ]:
import time
from functools import wraps


def timer(func):
    """计时装饰器：训练 step profiling 常用。"""
    @wraps(func)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        out = func(*args, **kwargs)
        elapsed = time.perf_counter() - start
        print(f"[{func.__name__}] elapsed={elapsed*1000:.2f}ms")
        return out
    return wrapper


@timer
def fake_train_step(size: int = 2000):
    x = np.random.randn(size, size)
    y = np.random.randn(size, size)
    z = x @ y
    return float(z.mean())

value = fake_train_step(400)
print("result:", round(value, 4))

In [ ]:
def text_stream(num_rows: int = 10000):
    """模拟大规模样本流式读取。"""
    for i in range(num_rows):
        yield {
            "instruction": f"Translate sentence {i}",
            "input": f"This is sentence number {i}.",
            "output": f"这是第 {i} 号句子。",
        }

# 只取前 5 条，不会把所有数据一次性载入内存
for idx, sample in enumerate(text_stream(1000000)):
    if idx < 5:
        print(sample)
    else:
        break

In [ ]:
# 用 pandas 快速构建训练样本 DataFrame
samples = list(text_stream(200))
df = pd.DataFrame(samples)

# 数据清洗
# 1) 去重
# 2) 过滤过短输出
# 3) 构造训练文本模板

df = df.drop_duplicates(subset=["instruction", "input"])
df = df[df["output"].str.len() >= 4]

def build_prompt(row):
    return (
        "<|im_start|>user\n"
        f"{row['instruction']}\n{row['input']}"
        "<|im_end|>\n"
        "<|im_start|>assistant\n"
        f"{row['output']}"
        "<|im_end|>"
    )

df["text"] = df.apply(build_prompt, axis=1)
df["char_len"] = df["text"].str.len()

print("shape:", df.shape)
print(df[["instruction", "char_len"]].head(3))
print("char_len describe:\n", df["char_len"].describe())

## Part B. 神经网络核心：从零手写前向/反向传播

这一部分我们用 NumPy 手写一个最小神经网络，重点观察：

- loss 如何从高到低。
- 梯度是不是算对（用数值梯度做校验）。
- 学习率变化对收敛速度和稳定性的影响。

In [ ]:
# 构造一个二分类数据集（二维点）
n_per_class = 200
mean0 = np.array([-1.2, -1.0])
mean1 = np.array([1.0, 1.2])
cov = np.array([[0.5, 0.2], [0.2, 0.5]])

x0 = np.random.multivariate_normal(mean0, cov, size=n_per_class)
x1 = np.random.multivariate_normal(mean1, cov, size=n_per_class)

X = np.vstack([x0, x1])  # (N, 2)
y = np.concatenate([np.zeros(n_per_class), np.ones(n_per_class)]).reshape(-1, 1)  # (N, 1)

perm = np.random.permutation(len(X))
X, y = X[perm], y[perm]

print("X shape:", X.shape, "y shape:", y.shape)

plt.figure(figsize=(5, 4))
plt.scatter(X[:, 0], X[:, 1], c=y.squeeze(), cmap="coolwarm", s=16)
plt.title("Synthetic Binary Classification Data")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

In [ ]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))


def bce_loss(pred, target, eps=1e-8):
    pred = np.clip(pred, eps, 1 - eps)
    return -np.mean(target * np.log(pred) + (1 - target) * np.log(1 - pred))


# 两层 MLP: 2 -> 16 -> 1
hidden_dim = 16
W1 = np.random.randn(2, hidden_dim) * 0.1
b1 = np.zeros((1, hidden_dim))
W2 = np.random.randn(hidden_dim, 1) * 0.1
b2 = np.zeros((1, 1))

lr = 0.08
epochs = 500
loss_curve = []
acc_curve = []

for epoch in range(epochs):
    # forward
    z1 = X @ W1 + b1
    a1 = np.tanh(z1)
    z2 = a1 @ W2 + b2
    pred = sigmoid(z2)

    loss = bce_loss(pred, y)
    loss_curve.append(loss)

    pred_label = (pred > 0.5).astype(np.float32)
    acc = (pred_label == y).mean()
    acc_curve.append(acc)

    # backward
    N = X.shape[0]
    dz2 = (pred - y) / N
    dW2 = a1.T @ dz2
    db2 = np.sum(dz2, axis=0, keepdims=True)

    da1 = dz2 @ W2.T
    dz1 = da1 * (1 - np.tanh(z1) ** 2)
    dW1 = X.T @ dz1
    db1 = np.sum(dz1, axis=0, keepdims=True)

    # gradient descent
    W1 -= lr * dW1
    b1 -= lr * db1
    W2 -= lr * dW2
    b2 -= lr * db2

    if (epoch + 1) % 100 == 0:
        print(f"epoch={epoch+1:03d}, loss={loss:.4f}, acc={acc:.4f}")

print("final loss:", round(loss_curve[-1], 4), "final acc:", round(acc_curve[-1], 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(loss_curve)
axes[0].set_title("Manual NN Loss")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("BCE loss")

axes[1].plot(acc_curve)
axes[1].set_title("Manual NN Accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("accuracy")
plt.tight_layout()
plt.show()

# 决策边界可视化
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

grid = np.c_[xx.ravel(), yy.ravel()]
z1_g = grid @ W1 + b1
a1_g = np.tanh(z1_g)
z2_g = a1_g @ W2 + b2
pred_g = sigmoid(z2_g).reshape(xx.shape)

plt.figure(figsize=(5, 4))
plt.contourf(xx, yy, pred_g > 0.5, alpha=0.3, cmap="coolwarm")
plt.scatter(X[:, 0], X[:, 1], c=y.squeeze(), s=12, cmap="coolwarm")
plt.title("Manual NN Decision Boundary")
plt.xlabel("x1")
plt.ylabel("x2")
plt.show()

In [ ]:
def forward_loss(W1_, b1_, W2_, b2_):
    z1_ = X @ W1_ + b1_
    a1_ = np.tanh(z1_)
    z2_ = a1_ @ W2_ + b2_
    pred_ = sigmoid(z2_)
    return bce_loss(pred_, y)

# 在当前点做一次解析梯度
z1 = X @ W1 + b1
a1 = np.tanh(z1)
z2 = a1 @ W2 + b2
pred = sigmoid(z2)

N = X.shape[0]
dz2 = (pred - y) / N
dW2 = a1.T @ dz2
db2 = np.sum(dz2, axis=0, keepdims=True)
da1 = dz2 @ W2.T
dz1 = da1 * (1 - np.tanh(z1) ** 2)
dW1 = X.T @ dz1
db1 = np.sum(dz1, axis=0, keepdims=True)

# 数值梯度检验：抽查一个参数
eps = 1e-5
i, j = 0, 0

W1_pos = W1.copy()
W1_neg = W1.copy()
W1_pos[i, j] += eps
W1_neg[i, j] -= eps

num_grad = (forward_loss(W1_pos, b1, W2, b2) - forward_loss(W1_neg, b1, W2, b2)) / (2 * eps)
ana_grad = dW1[i, j]

print("numerical grad:", num_grad)
print("analytic grad :", ana_grad)
print("abs diff      :", abs(num_grad - ana_grad))

## Part C. PyTorch 基础到训练闭环

下面把刚才 NumPy 版本迁移到 PyTorch，重点体会：

- Tensor 形状操作如何映射到模型结构。
- Autograd 如何自动计算梯度。
- Dataset/DataLoader/训练循环如何组织成工程化代码。

In [ ]:
# 形状操作演示（模拟 attention 的张量变形）
B, L, D, H = 2, 5, 16, 4
x = torch.randn(B, L, D)
print("x:", x.shape)

d_head = D // H
x_heads = x.view(B, L, H, d_head).transpose(1, 2)
print("x_heads (B, H, L, d_h):", x_heads.shape)

# 再拼回去
x_back = x_heads.transpose(1, 2).contiguous().view(B, L, D)
print("x_back:", x_back.shape)
print("allclose:", torch.allclose(x, x_back))

In [ ]:
# autograd 示例：y = x^2 + 3x
x = torch.tensor([2.0, 3.0], requires_grad=True)
y = x ** 2 + 3 * x
loss = y.sum()
loss.backward()

print("x:", x)
print("y:", y)
print("grad:", x.grad)  # 理论值: 2x + 3 -> [7, 9]

# 梯度清零
x.grad.zero_()
print("grad after zero:", x.grad)

In [ ]:
class BinaryToyDataset(Dataset):
    def __init__(self, X_np: np.ndarray, y_np: np.ndarray):
        self.X = torch.tensor(X_np, dtype=torch.float32)
        self.y = torch.tensor(y_np, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]


# 训练/验证切分
split = int(0.8 * len(X))
X_train, y_train = X[:split], y[:split]
X_val, y_val = X[split:], y[split:]

train_ds = BinaryToyDataset(X_train, y_train)
val_ds = BinaryToyDataset(X_val, y_val)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)

print("train size:", len(train_ds), "val size:", len(val_ds))

In [ ]:
class MLPBinaryClassifier(nn.Module):
    def __init__(self, in_dim=2, hidden=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.ReLU(),
            nn.Linear(hidden, hidden),
            nn.ReLU(),
            nn.Linear(hidden, 1),
        )

    def forward(self, x):
        return self.net(x)


def evaluate(model, loader):
    model.eval()
    total_loss = 0.0
    total = 0
    correct = 0
    criterion = nn.BCEWithLogitsLoss()

    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = criterion(logits, yb)

            pred = (torch.sigmoid(logits) > 0.5).float()
            correct += (pred == yb).sum().item()
            total += yb.numel()
            total_loss += loss.item() * xb.size(0)

    return total_loss / len(loader.dataset), correct / total


model = MLPBinaryClassifier().to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
criterion = nn.BCEWithLogitsLoss()

num_epochs = 50
train_losses, val_losses, val_accs = [], [], []

for epoch in range(num_epochs):
    model.train()
    running = 0.0

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        logits = model(xb)
        loss = criterion(logits, yb)

        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        running += loss.item() * xb.size(0)

    train_loss = running / len(train_loader.dataset)
    val_loss, val_acc = evaluate(model, val_loader)

    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accs.append(val_acc)

    if (epoch + 1) % 10 == 0:
        print(f"epoch={epoch+1:02d} train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

print("final val acc:", round(val_accs[-1], 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
axes[0].plot(train_losses, label="train")
axes[0].plot(val_losses, label="val")
axes[0].set_title("PyTorch Training Curve")
axes[0].set_xlabel("epoch")
axes[0].set_ylabel("BCE loss")
axes[0].legend()

axes[1].plot(val_accs)
axes[1].set_title("Validation Accuracy")
axes[1].set_xlabel("epoch")
axes[1].set_ylabel("acc")
plt.tight_layout()
plt.show()

# 手写 confusion matrix
model.eval()
all_pred, all_true = [], []
with torch.no_grad():
    for xb, yb in val_loader:
        logits = model(xb.to(DEVICE))
        pred = (torch.sigmoid(logits) > 0.5).long().cpu().numpy().reshape(-1)
        true = yb.long().cpu().numpy().reshape(-1)
        all_pred.extend(pred.tolist())
        all_true.extend(true.tolist())

all_pred = np.array(all_pred)
all_true = np.array(all_true)

tp = ((all_pred == 1) & (all_true == 1)).sum()
tn = ((all_pred == 0) & (all_true == 0)).sum()
fp = ((all_pred == 1) & (all_true == 0)).sum()
fn = ((all_pred == 0) & (all_true == 1)).sum()

print("Confusion Matrix")
print(np.array([[tn, fp], [fn, tp]]))
print("precision:", round(tp / max(tp + fp, 1), 4))
print("recall   :", round(tp / max(tp + fn, 1), 4))

## Part D. 从分类到序列建模：一个最小 next-token 训练示例

为了贴近 LLM 场景，我们再做一个超小字符级语言模型：

- 自建字符词表。
- 用 `Embedding + GRU + Linear` 预测下一个字符。
- 观察 loss 下降和生成效果变化。

In [ ]:
corpus = (
    "deep learning is fun. "
    "pytorch makes training easier. "
    "we learn by coding and debugging. "
    "gradient descent updates parameters. "
) * 80

chars = sorted(list(set(corpus)))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
vocab_size = len(chars)

encoded = torch.tensor([stoi[c] for c in corpus], dtype=torch.long)

seq_len = 24
X_seq, y_seq = [], []
for i in range(len(encoded) - seq_len):
    X_seq.append(encoded[i : i + seq_len])
    y_seq.append(encoded[i + 1 : i + seq_len + 1])

X_seq = torch.stack(X_seq)
y_seq = torch.stack(y_seq)

split_seq = int(0.9 * len(X_seq))
X_train_seq, y_train_seq = X_seq[:split_seq], y_seq[:split_seq]
X_val_seq, y_val_seq = X_seq[split_seq:], y_seq[split_seq:]

print("vocab_size:", vocab_size, "train_seq:", X_train_seq.shape, "val_seq:", X_val_seq.shape)


class CharSeqDataset(Dataset):
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __len__(self):
        return len(self.x)

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]


train_seq_loader = DataLoader(CharSeqDataset(X_train_seq, y_train_seq), batch_size=64, shuffle=True)
val_seq_loader = DataLoader(CharSeqDataset(X_val_seq, y_val_seq), batch_size=128)


class TinyCharLM(nn.Module):
    def __init__(self, vocab_size, emb_dim=32, hidden_dim=64):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, emb_dim)
        self.gru = nn.GRU(emb_dim, hidden_dim, batch_first=True)
        self.head = nn.Linear(hidden_dim, vocab_size)

    def forward(self, x):
        e = self.embed(x)
        h, _ = self.gru(e)
        logits = self.head(h)
        return logits


def eval_lm(model, loader):
    model.eval()
    total_loss = 0.0
    total_tokens = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            logits = model(xb)
            loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1), reduction="sum")
            total_loss += loss.item()
            total_tokens += yb.numel()
    return total_loss / total_tokens


lm = TinyCharLM(vocab_size).to(DEVICE)
opt = torch.optim.AdamW(lm.parameters(), lr=3e-3)

lm_train_curve, lm_val_curve = [], []
for epoch in range(20):
    lm.train()
    for xb, yb in train_seq_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        logits = lm(xb)
        loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))

        opt.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(lm.parameters(), 1.0)
        opt.step()

    train_loss = eval_lm(lm, train_seq_loader)
    val_loss = eval_lm(lm, val_seq_loader)
    lm_train_curve.append(train_loss)
    lm_val_curve.append(val_loss)
    print(f"epoch={epoch+1:02d}, train_nll={train_loss:.4f}, val_nll={val_loss:.4f}")

In [ ]:
plt.figure(figsize=(5, 3.5))
plt.plot(lm_train_curve, label="train")
plt.plot(lm_val_curve, label="val")
plt.title("Tiny Char LM NLL")
plt.xlabel("epoch")
plt.ylabel("nll")
plt.legend()
plt.show()


def generate_text(model, start: str, max_new_tokens: int = 80, temperature: float = 1.0):
    model.eval()
    x = torch.tensor([[stoi[ch] for ch in start]], dtype=torch.long, device=DEVICE)

    with torch.no_grad():
        for _ in range(max_new_tokens):
            logits = model(x)
            next_logits = logits[:, -1, :] / temperature
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)
            x = torch.cat([x, next_token], dim=1)

    out = "".join(itos[idx] for idx in x[0].tolist())
    return out

print(generate_text(lm, start="deep ", max_new_tokens=120, temperature=0.9))

## Part E. 训练模板与工程习惯

下面给一个你可以直接复制到项目里的训练模板：

- dataclass 管理超参。
- 支持 AMP 混合精度。
- 统一日志输出。
- 简单 Early Stopping。

In [ ]:
@dataclass
class TrainConfig:
    lr: float = 2e-3
    weight_decay: float = 1e-3
    epochs: int = 30
    grad_clip: float = 1.0
    use_amp: bool = True
    early_stop_patience: int = 5


def train_with_template(model, train_loader, val_loader, cfg: TrainConfig):
    model = model.to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=cfg.lr, weight_decay=cfg.weight_decay)
    criterion = nn.BCEWithLogitsLoss()
    scaler = torch.amp.GradScaler(enabled=(cfg.use_amp and DEVICE == "cuda"))

    best_val = float("inf")
    bad_epochs = 0

    for epoch in range(cfg.epochs):
        model.train()
        total = 0.0

        for xb, yb in train_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()

            autocast_enabled = cfg.use_amp and DEVICE == "cuda"
            with torch.amp.autocast(device_type=DEVICE, enabled=autocast_enabled):
                logits = model(xb)
                loss = criterion(logits, yb)

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
            scaler.step(optimizer)
            scaler.update()
            total += loss.item() * xb.size(0)

        train_loss = total / len(train_loader.dataset)
        val_loss, val_acc = evaluate(model, val_loader)
        print(f"[template] epoch={epoch+1:02d} train_loss={train_loss:.4f} val_loss={val_loss:.4f} val_acc={val_acc:.4f}")

        # early stopping
        if val_loss < best_val:
            best_val = val_loss
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= cfg.early_stop_patience:
                print("Early stopping triggered.")
                break

    return model


_ = train_with_template(MLPBinaryClassifier(), train_loader, val_loader, TrainConfig())

## Part F. 练习题（建议你亲手做）

### 练习 1：学习率实验

把 Part B 中 `lr` 分别改成 `0.005 / 0.05 / 0.5`，比较：

- 收敛速度。
- 最终精度。
- 是否震荡或发散。

### 练习 2：激活函数对比

把 Part C 的 `ReLU` 换成 `Tanh / GELU / SiLU`，观察：

- 训练 loss 曲线是否更平滑。
- 验证集精度是否更高。

### 练习 3：模型容量实验

将 `hidden` 改为 `8 / 16 / 64 / 128`，比较 bias-variance：

- 小模型是否欠拟合。
- 大模型是否更容易过拟合。

### 练习 4：过拟合抑制

尝试添加：

- `Dropout(0.2)`
- 更强 `weight_decay`
- Early stopping patience 更小

记录最佳组合。

## 你现在应该掌握的能力

完成本 Notebook 后，你应当能做到：

- 用 Python 把数据处理流程写成可复用函数和生成器。
- 手写一个最小神经网络并验证反向传播正确性。
- 用 PyTorch 从零搭建 `Dataset → DataLoader → Model → Train/Eval` 闭环。
- 看懂 loss/acc 曲线并做基础调参。

如果你愿意，我下一步可以继续补：

1. `CNN + CIFAR10` 基础视觉训练 Notebook。
2. `Transformer mini` 从零实现 Notebook。
3. `PyTorch 调试实战`（梯度爆炸、NaN、显存 OOM 排查）Notebook。